# VoiceEye FastLane — MLOps Training Pipeline

This notebook trains a YOLO26n object detection model for VoiceEye's Fast Lane,
with full ClearML experiment tracking, model evaluation, and ONNX export.

**Before starting:**
- Run `clearml-init` to configure your ClearML backend credentials.
- Place `yolo26n.pt` base weights in the project root (or adjust `config.yaml`).
- Ensure a GPU is available for training.

All hyperparameters are loaded from `training/config.yaml` — the single source of truth.

## 1. Setup Environment

In [ ]:
!pip install -r training/requirements.txt -q

## 2. Load Configuration

All training parameters come from `training/config.yaml`. You can override
individual values in the cell below without editing the YAML file.

In [ ]:
import yaml
from pathlib import Path

with open("training/config.yaml") as f:
    cfg = yaml.safe_load(f)

# ── Optional overrides (uncomment to change) ──
# cfg["epochs"] = 200
# cfg["batch"] = 32
# cfg["dataset_id"] = "your-new-dataset-id"

# Display active config
print("Active Configuration:")
print("=" * 50)
for key, value in cfg.items():
    print(f"  {key}: {value}")
print("=" * 50)

## 3. Initialize ClearML Tracking

The config dict is connected as hyperparameters — visible and editable in the ClearML Web UI.

In [ ]:
from clearml import Task, Dataset

task = Task.init(
    project_name=cfg["clearml_project"],
    task_name=cfg["clearml_task_name"],
)
task.connect(cfg, name="training_config")
print(f"ClearML Task ID: {task.id}")
print(f"View at: {task.get_output_log_web_page()}")

## 4. Fetch Dataset

Downloads the dataset from ClearML using the ID in config.

**To push a new dataset**, uncomment the Dataset Update Block below.

In [ ]:
# ── DATASET FETCH (default) ──
dataset = Dataset.get(dataset_id=cfg["dataset_id"])
dataset_path = dataset.get_local_copy()
print(f"Dataset downloaded to: {dataset_path}")

# Verify data.yaml exists
data_yaml = Path(dataset_path) / "data.yaml"
assert data_yaml.exists(), f"data.yaml not found at {data_yaml}"

# Show dataset info
with open(data_yaml) as f:
    data_info = yaml.safe_load(f)
print(f"\nClasses ({len(data_info.get('names', []))}): {data_info.get('names', [])}")

# ── DATASET UPDATE BLOCK (uncomment to push a NEW dataset) ──
# new_dataset = Dataset.create(
#     dataset_project="VoiceEye",
#     dataset_name="FastLaneData_v2"
# )
# new_dataset.add_files(path="/path/to/your/updated/local/dataset")
# new_dataset.upload()
# new_dataset.finalize()
# dataset_path = new_dataset.get_local_copy()
# print(f"New dataset ID: {new_dataset.id}")

## 5. Train YOLO26n

Ultralytics automatically logs loss curves, mAP metrics, and sample images
to the active ClearML task. Training uses cosine annealing LR with early stopping.

In [ ]:
from ultralytics import YOLO

model = YOLO(cfg["model_weights"])

results = model.train(
    data=str(data_yaml),
    epochs=cfg["epochs"],
    patience=cfg["patience"],
    batch=cfg["batch"],
    imgsz=cfg["imgsz"],
    optimizer=cfg["optimizer"],
    lr0=cfg["lr0"],
    lrf=cfg["lrf"],
    cos_lr=cfg["cos_lr"],
    warmup_epochs=cfg["warmup_epochs"],
    warmup_momentum=cfg["warmup_momentum"],
    weight_decay=cfg["weight_decay"],
    dropout=cfg["dropout"],
    hsv_h=cfg["hsv_h"],
    hsv_s=cfg["hsv_s"],
    hsv_v=cfg["hsv_v"],
    degrees=cfg["degrees"],
    translate=cfg["translate"],
    scale=cfg["scale"],
    fliplr=cfg["fliplr"],
    mosaic=cfg["mosaic"],
    mixup=cfg["mixup"],
    project="VoiceEye_Runs",
    name="fastlane_train",
    exist_ok=True,
)

print("\nTraining complete!")
print(f"Best weights: {model.trainer.best}")

## 6. Evaluate Model

Run validation and check against the quality gate threshold.

In [ ]:
metrics = model.val(
    data=str(data_yaml),
    conf=cfg["conf_threshold"],
    iou=cfg["iou_threshold"],
)

map50 = float(metrics.box.map50)
map50_95 = float(metrics.box.map)
precision = float(metrics.box.mp)
recall = float(metrics.box.mr)

print(f"\n{'='*50}")
print(f"  mAP@50:    {map50:.4f}")
print(f"  mAP@50-95: {map50_95:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"{'='*50}")

# Quality gate
min_map = cfg.get("min_map50", 0.40)
if map50 >= min_map:
    print(f"\n  QUALITY GATE PASSED (mAP@50 = {map50:.4f} >= {min_map:.2f})")
else:
    print(f"\n  QUALITY GATE FAILED (mAP@50 = {map50:.4f} < {min_map:.2f})")
    print("  Consider more epochs, different augmentation, or HPO.")

# Report to ClearML for HPO objective
task.get_logger().report_scalar(
    title="val", series="mAP50",
    value=map50, iteration=cfg["epochs"]
)

# Per-class AP
print("\nPer-class AP@50:")
if hasattr(metrics.box, 'ap50') and data_info.get('names'):
    names = data_info['names']
    if isinstance(names, dict):
        names = list(names.values())
    for i, ap in enumerate(metrics.box.ap50):
        if i < len(names):
            print(f"  {names[i]:20s} {float(ap):.4f}")

## 7. Compare with Previous Best

Query ClearML for the most recent completed training task and compare metrics.

In [ ]:
from clearml import Task as TaskQuery

# Find previous completed tasks in the same project
prev_tasks = TaskQuery.get_tasks(
    project_name=cfg["clearml_project"],
    task_name=cfg["clearml_task_name"],
    task_filter={"status": ["completed"]},
    order_by=["-last_update"],
)

# Exclude current task
prev_tasks = [t for t in prev_tasks if t.id != task.id]

if prev_tasks:
    prev = prev_tasks[0]
    prev_scalars = prev.get_last_scalar_metrics()
    prev_map50 = None
    if "val" in prev_scalars and "mAP50" in prev_scalars["val"]:
        prev_map50 = prev_scalars["val"]["mAP50"]["last"]

    print(f"{'Metric':<20} {'Previous':>12} {'Current':>12} {'Delta':>12}")
    print("-" * 56)
    if prev_map50 is not None:
        delta = map50 - prev_map50
        sign = "+" if delta >= 0 else ""
        print(f"{'mAP@50':<20} {prev_map50:>12.4f} {map50:>12.4f} {sign}{delta:>11.4f}")
    else:
        print(f"  Could not retrieve previous mAP@50 from task {prev.id}")
    print(f"\nPrevious task: {prev.id} ({prev.name})")
else:
    print("No previous completed tasks found — this is the first run.")

## 8. Export to ONNX

Export for ONNX Runtime Web (WASM backend, float32, opset 17).

In [ ]:
import hashlib

export_path = model.export(
    format=cfg["export_format"],
    imgsz=cfg["export_imgsz"],
    half=cfg["half"],
    simplify=cfg.get("export_simplify", True),
    opset=cfg.get("export_opset", 17),
)

onnx_path = Path(export_path)
size_mb = onnx_path.stat().st_size / (1024 * 1024)

# Compute SHA-256 checksum
sha256 = hashlib.sha256()
with open(onnx_path, "rb") as f:
    for chunk in iter(lambda: f.read(8192), b""):
        sha256.update(chunk)
checksum = sha256.hexdigest()

print(f"Exported: {onnx_path}")
print(f"Size:     {size_mb:.2f} MB")
print(f"SHA-256:  {checksum}")
print(f"\nCopy this file to public/models/yolo26n.onnx for deployment.")

## 9. Register Artifact & Close

Upload the ONNX model to ClearML with full metadata for traceability.

In [ ]:
task.upload_artifact(
    name="fastlane_onnx_model",
    artifact_object=str(onnx_path),
    metadata={
        "map50": map50,
        "map50_95": map50_95,
        "precision": precision,
        "recall": recall,
        "dataset_id": cfg["dataset_id"],
        "epochs": cfg["epochs"],
        "optimizer": cfg["optimizer"],
        "lr0": cfg["lr0"],
        "model_size_mb": round(size_mb, 2),
        "sha256": checksum,
    },
)
print("ONNX model uploaded to ClearML artifacts.")

task.close()
print("\nClearML task closed. Training pipeline complete!")